# WEEX V6.49 — MARKET i LIMIT
Jedna komorka. Wyniki ZIP pobieraja sie automatycznie. Skan nie sklada zlecen.

In [ ]:
import requests
import shutil
from pathlib import Path
from google.colab import drive, files

# Historia OI pozostaje wspolna miedzy wersjami
drive.mount("/content/drive")
oi = Path("/content/drive/MyDrive/WEEX")
oi.mkdir(parents=True, exist_ok=True)
state = oi / "oi_history_shared.json"
if not state.exists():
    old = [oi / f"weex_v{v}_oi_history_state.json" for v in (648,647,646,645,644)]
    existing = [p for p in old if p.exists()]
    if existing:
        shutil.copy2(max(existing,key=lambda p:p.stat().st_mtime), state)

# Pobierz najnowszy kolektor
url = "https://raw.githubusercontent.com/danieldziezak1986-hash/weex-futures/main/weex_v649_collector.py"
r = requests.get(url, timeout=30)
r.raise_for_status()
Path("weex_v649_collector.py").write_bytes(r.content)

# Testy wewnetrzne
%run weex_v649_collector.py --self-test

# TOP 150 Crypto + TOP 50 TradFi, MARKET + LIMIT
%run weex_v649_collector.py --top 150 --tradfi-top 50 --max-deep 30 --limit-ttl-minutes 180 --state-path /content/drive/MyDrive/WEEX/oi_history_shared.json

# Pobierz najnowsze wyniki ZIP
folder = Path("/content/weex_v649_results")
archives = list(folder.glob("weex_v649_*.zip"))
if not archives:
    raise FileNotFoundError("Nie znaleziono ZIP WEEX V6.49")
latest = max(archives, key=lambda p:p.stat().st_mtime)
print("Pobieranie wynikow:",latest.name)
files.download(str(latest))
